In [1]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.rag.document_chunker import HybridDocumentChunker

# 1. تهيئة المسارات وتحميل الكلاس
data_path = "../data_Json/processed/master_scholarships_clean.json"
chunker = HybridDocumentChunker(chunk_size=500, chunk_overlap=50)

# 2. تنفيذ الخطوة الأولى: التحميل والتصفية
print(">>> Loading and Filtering Documents...")
base_docs = chunker.load_and_filter(data_path)
print(f"Loaded {len(base_docs)} base documents successfully.")

# 3. تنفيذ الخطوة الثانية: التقطيع الهجين
print(">>> Applying Hybrid Chunking...")
final_chunks = chunker.process_documents(base_docs)
print(f"Generated {len(final_chunks)} total chunks.")

# 4. التحليل الهندسي المتقدم (Advanced Diagnostics)
print("\n--- Chunking Quality Diagnostics ---")

# حساب أطوال القطع
chunk_lengths = [len(chunk.page_content) for chunk in final_chunks]
df_stats = pd.Series(chunk_lengths)

print("Character Length Distribution:")
print(df_stats.describe())

# فحص القطع المتيتّمة (Chunks missing crucial semantic context)
orphan_chunks = 0
for chunk in final_chunks:
    if 'Document_Title' not in chunk.metadata and 'Section_Title' not in chunk.metadata:
        orphan_chunks += 1

print(f"\nOrphan Chunks (Missing Contextual Headers): {orphan_chunks}")

# فحص تسوية البيانات الوصفية (Metadata Flattening Validation)
nested_metadata = 0
for chunk in final_chunks:
    if any(isinstance(val, dict) for val in chunk.metadata.values()):
        nested_metadata += 1

print(f"Chunks with nested metadata (ChromaDB Crash Risk): {nested_metadata}")

# 5. عرض عينة عشوائية لفحص جودة التقطيع
import random
print("\n>>> Random Chunk Inspection <<<")
sample_chunk = random.choice(final_chunks)
print("METADATA:")
for k, v in sample_chunk.metadata.items():
    print(f"  {k}: {v}")
print("\nCONTENT:")
print("-" * 50)
print(sample_chunk.page_content)
print("-" * 50)

>>> Loading and Filtering Documents...
Loaded 2526 base documents successfully.
>>> Applying Hybrid Chunking...
Generated 20440 total chunks.

--- Chunking Quality Diagnostics ---
Character Length Distribution:
count    20440.000000
mean       357.842319
std        123.387443
min         39.000000
25%        269.000000
50%        399.000000
75%        461.000000
max        500.000000
dtype: float64

Orphan Chunks (Missing Contextual Headers): 0
Chunks with nested metadata (ChromaDB Crash Risk): 0

>>> Random Chunk Inspection <<<
METADATA:
  scholarship_name: CBC Spouses Visual Arts Scholarship
  host_country: USA
  eligible_nationality: International / All
  academic_level: Undergraduates
  academic_major: architecture, ceramics, drawing, fashion, graphic design, illustration, painting, photography, video production, other decorative arts
  funding_category: Fixed Grant
  funding_amount: $5,000
  standardized_deadline: 2027-03-27
  scholarship_status: Active
  application_link: https:/

In [2]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# 1. تحميل نموذج التضمين متعدد اللغات
embedding_model_name = "BAAI/bge-m3"
print(f">>> Loading Embedding Model: {embedding_model_name}...")

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)


print("Embedding Model Loaded Successfully.")
print("Model Context Window: 8192 tokens (~32,000 characters).")
print("Architectural Ceiling: Safe chunk sizes can range from 400 to 2000 characters effortlessly.")

>>> Loading Embedding Model: BAAI/bge-m3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding Model Loaded Successfully.
Model Context Window: 8192 tokens (~32,000 characters).
Architectural Ceiling: Safe chunk sizes can range from 400 to 2000 characters effortlessly.


In [3]:
import random

# 1. إنشاء عينة تمثيلية (Representative Sample) من 5 منح متنوعة
# نختار منحاً قصيرة، وطويلة (مثل DAAD)، ومتعددة اللغات
random.seed(42)
sample_docs = random.sample(base_docs, min(5, len(base_docs)))

print(f">>> Selected {len(sample_docs)} representative scholarship documents for tuning.\n")

# 2. مصفوفة التكوينات لتجربتها
experiment_matrix = [
    {"name": "Strict/Fine-Grained", "size": 400, "overlap": 50},
    {"name": "Balanced (Recommended)", "size": 800, "overlap": 100},
    {"name": "Broad Context", "size": 1200, "overlap": 150}
]


for config in experiment_matrix:
    print(f"==================================================")
    print(f" TESTING CONFIGURATION: {config['name']}")
    print(f" Chunk Size: {config['size']} | Overlap: {config['overlap']}")
    print(f"==================================================")
    
    test_chunker = HybridDocumentChunker(
        chunk_size=config['size'],
        chunk_overlap=config['overlap'],
        min_chunk_length=30
    )
    
    sample_chunks = test_chunker.process_documents(sample_docs)
    print(f"Total Chunks Generated for Sample: {len(sample_chunks)}")
    

    print("\n--- Sample Chunk Preview ---")
    print(f"Content:\n{sample_chunks[0].page_content[:300]}...")
    print(f"Metadata Header: {sample_chunks[0].metadata.get('Section_Title', 'N/A')}\n")

>>> Selected 5 representative scholarship documents for tuning.

 TESTING CONFIGURATION: Strict/Fine-Grained
 Chunk Size: 400 | Overlap: 50
Total Chunks Generated for Sample: 77

--- Sample Chunk Preview ---
Content:
# Scholarship: Calculated Genius STEMinist Scholarship  
## Program Description...
Metadata Header: Program Description

 TESTING CONFIGURATION: Balanced (Recommended)
 Chunk Size: 800 | Overlap: 100
Total Chunks Generated for Sample: 38

--- Sample Chunk Preview ---
Content:
# Scholarship: Calculated Genius STEMinist Scholarship  
## Program Description
The Calculated Genius STEMINIST Scholarship Program provides financial support to Chicago metropolitan area women-of-color pursuing college degrees in engineering. Calculated Genius is proud to support and encourage wome...
Metadata Header: Program Description

 TESTING CONFIGURATION: Broad Context
 Chunk Size: 1200 | Overlap: 150
Total Chunks Generated for Sample: 23

--- Sample Chunk Preview ---
Content:
# Scholarship: C

In [4]:

results = []

for config in experiment_matrix:
    test_chunker = HybridDocumentChunker(
        chunk_size=config['size'],
        chunk_overlap=config['overlap'],
        min_chunk_length=30
    )
    all_test_chunks = test_chunker.process_documents(sample_docs)
    
    lengths = [len(c.page_content) for c in all_test_chunks]
    avg_len = sum(lengths) / len(lengths) if lengths else 0
    
    results.append({
        "Configuration": config['name'],
        "Chunk Size": config['size'],
        "Overlap": config['overlap'],
        "Total Chunks": len(all_test_chunks),
        "Avg Chunk Length": round(avg_len, 2)
    })


import pandas as pd
df_results = pd.DataFrame(results)
print(">>> Hyperparameter Tuning Benchmark Summary <<<")
print(df_results.to_string(index=False))

>>> Hyperparameter Tuning Benchmark Summary <<<
         Configuration  Chunk Size  Overlap  Total Chunks  Avg Chunk Length
   Strict/Fine-Grained         400       50            77            264.79
Balanced (Recommended)         800      100            38            551.74
         Broad Context        1200      150            23            909.00
